In [2]:
from datasets import load_dataset

imdb_dataset = load_dataset("imdb")
split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_valid_set = split["train"], split["test"]
imdb_test_set = imdb_dataset["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
imdb_train_set[1]["text"]

"'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!"

In [4]:
imdb_train_set[1]["label"]

1

In [5]:
imdb_train_set[15]
#Each set contains text and label which is positive for 1 and negative for 0

{'text': 'My wife, Kate and I absolutely loved the series and can\'t wait for the next one (hopefully there is a sequel!). I would love to know what the catchy song is called and who wrote it, maybe because I am "old and grey" and still interested in life:-). If anyone has the full lyrics please send them.<br /><br />Of course one big reason why my wife and I liked this series so much was that we are 75 years old and retired but still very active intellectually. It\'s great to see a show that highlights the contribution to society that can still be made by older people with special skills and experience. The human interest aspect showing the interactions of the characters and the younger people around them is an important part of the show.<br /><br />This series is highly entertaining and very sophisticated, on a par with some of our other favourites, "Spooks" and "Hustle".',
 'label': 1}

In [6]:
import tokenizers

bpe_model = tokenizers.models.BPE(unk_token="<unk>") # to declare unknown words as <unk>
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<pad>", "<unk>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000,
                                             special_tokens=special_tokens)
train_reviews = [review["text"].lower() for review in imdb_train_set]
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [7]:
new_review = "What a fantastic movie😍"

In [8]:
bpe_encoding = bpe_tokenizer.encode(new_review)
bpe_encoding

Encoding(num_tokens=10, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [9]:
bpe_encoding.tokens

# Here we can see that the bpe model learned new subtokens from the training set text and also fill the unknown text with <unk>
# We set the vocab size as 1000 so it can store upto 1000  new subtokens to find words which contain the subtoken

['<unk>', 'h', 'at', 'a', 'f', 'ant', 'ast', 'ic', 'movie', '<unk>']

In [10]:
bpe_token_ids = bpe_encoding.ids
bpe_token_ids
# The ids that assigned to every sub token after encoding

[1, 49, 144, 42, 47, 279, 302, 171, 211, 1]

In [11]:
bpe_tokenizer.decode(bpe_token_ids)

'h at a f ant ast ic movie'

In [12]:
bpe_encoding.offsets
# The offset parameter will hold the words starting and finishing number as same for the word which are splitted during encoding

[(0, 1),
 (1, 2),
 (2, 4),
 (5, 6),
 (7, 8),
 (8, 11),
 (11, 14),
 (14, 16),
 (17, 22),
 (22, 23)]

In [13]:
bpe_tokenizer.encode_batch(train_reviews[:3])

[Encoding(num_tokens=281, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=114, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=285, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [14]:
bpe_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)
# we encode the batch using fixed token count so the remanining tokens were by default used <pad>

In [15]:
import torch

bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])
bpe_batch_ids = torch.tensor([encoding.ids for encoding in bpe_encodings])
bpe_batch_ids

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

In [16]:
attention_mask = torch.tensor([encoding.attention_mask for encoding in bpe_encodings])
attention_mask

# Attention mask will assign each token attention if the token is padded as 0 then it leaves it and if it is 1 then it uses for ignoring padded tokens

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [17]:
lengths = attention_mask.sum(dim = -1)
lengths

tensor([281, 114, 285])

In [18]:
tokenizers.pre_tokenizers.ByteLevel().pre_tokenize_str(new_review)
# By using Bytelevel Encoding it will not ignore space by adding Ġ instead and also assign some characters for emojis as it uses UTF-8 character encoding

[('ĠWhat', (0, 4)),
 ('Ġa', (4, 6)),
 ('Ġfantastic', (6, 16)),
 ('Ġmovie', (16, 22)),
 ('ðŁĺį', (22, 23))]

In [19]:
bbpe_model = tokenizers.models.BPE(unk_token = "<unk>")
bbpe_tokenizer = tokenizers.Tokenizer(bbpe_model)
bbpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
bbpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000, special_tokens=special_tokens)
bbpe_tokenizer.train_from_iterator(train_reviews, bbpe_trainer)

# Here we used pre_tokenizers.ByteLevel() instead of whitespace so the whitespace not ignored at tokenizer partitioning

In [20]:
bbpe_encoding = bbpe_tokenizer.encode(new_review)
bbpe_tokens = bbpe_encoding.tokens
bbpe_tokens

#Byte-Level Byte Pair encoding helps whenever there is different languages or characters in text so using Bytes to encode is the best solution

['Ġ',
 '<unk>',
 'h',
 'at',
 'Ġa',
 'Ġf',
 'ant',
 'ast',
 'ic',
 'Ġmovie',
 '<unk>',
 'Ł',
 'ĺ',
 'į']

In [21]:
bbpe_token_ids = bbpe_encoding.ids
bbpe_token_ids

[107, 1, 47, 145, 129, 142, 294, 318, 172, 232, 1, 125, 119, 112]

In [22]:
bbpe_decoded = bbpe_tokenizer.decode(bbpe_token_ids)
bbpe_decoded

'Ġ h at Ġa Ġf ant ast ic Ġmovie Ł ĺ į'

In [23]:
bbpe_decoded.replace(" ", "").replace("Ġ", " ").strip()

'hat a fantastic movieŁĺį'

In [24]:
wp_model = tokenizers.models.WordPiece(unk_token = "<unk>")
wp_tokenizer = tokenizers.Tokenizer(wp_model)
wp_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
wp_trainer = tokenizers.trainers.WordPieceTrainer(vocab_size = 1000, special_tokens = special_tokens)
wp_tokenizer.train_from_iterator(train_reviews, wp_trainer)

In [25]:
wp_encoding = wp_tokenizer.encode(new_review)
wp_tokens = wp_encoding.tokens
wp_tokens

# Word Piece Encoding uses probability calculations scores to determine the importance of tokens to add in vocabulary and it uses ## to join splitted words

['<unk>', 'a', 'fa', '##n', '##t', '##ast', '##ic', 'movie', '<unk>']

In [26]:
wp_token_ids = wp_encoding.ids
wp_token_ids

[1, 42, 551, 144, 156, 413, 279, 331, 1]

In [27]:
wp_decoded = wp_tokenizer.decode(wp_token_ids)
wp_decoded

'a fa ##n ##t ##ast ##ic movie'

In [28]:
wp_decoded.replace(" ##", "").replace(" !", "!")

'a fantastic movie'

In [29]:
unigram_model = tokenizers.models.Unigram()
unigram_tokenizer = tokenizers.Tokenizer(unigram_model)
unigram_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
unigram_trainer = tokenizers.trainers.UnigramTrainer(vocab_size = 1000, special_tokens = special_tokens, unk_token = "<unk>")
unigram_tokenizer.train_from_iterator(train_reviews, unigram_trainer)

In [30]:
unigram_encoding = unigram_tokenizer.encode(new_review)
unigram_tokens = unigram_encoding.tokens
unigram_tokens

#Unlike other encoding techniques it uses top down approach by removing tokens from whole text to check the impact and add it to vocab

['W', 'h', 'a', 't', 'a', 'fantastic', 'movie', '😍']

In [31]:
unigram_token_ids = unigram_encoding.ids[:10]
unigram_token_ids

[1, 34, 4, 3, 4, 821, 46, 1]

In [32]:
unigram_tokenizer.decode(unigram_token_ids)

'h a t a fantastic movie'

In [33]:
unigram_tokenizer.decode(unigram_token_ids, skip_special_tokens = False)

#By default the decoders uses the skip_special_tokens param as True so unrecognised tokens get ignored during decoding and here capital W didnt used in training data so it ignored

'<unk> h a t a fantastic movie <unk>'

In [34]:
import transformers

gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_encoding = gpt2_tokenizer(train_reviews[:3], truncation = True, max_length = 500)
# we used gpt_tokenizer which also uses the same as BBPE encoding to encode

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [35]:
gpt2_token_ids = gpt2_encoding["input_ids"][1][:10]
gpt2_token_ids

[470, 258, 12302, 6, 373, 257, 7932, 3807, 546, 262]

In [36]:
gpt2_tokenizer.decode(gpt2_token_ids)

"'the rookie' was a wonderful movie about the"

In [37]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoding = bert_tokenizer(train_reviews[:5], padding = True, truncation = True,
                               max_length = 500, return_tensors = "pt")
bert_encoding

# This BERT tokenizer uses the wordpiece encoding to encode the text and here we used padding as True fill the remain tokens as <unk> if there is none to match max length ogf the given review
# We used padding because here it returns python tensors instead of list which helps in nn tensors by using return_tensors = "pt"

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'input_ids': tensor([[  101,  2754, 17241,  ...,     0,     0,     0],
        [  101,  1005,  1996,  ...,     0,     0,     0],
        [  101,  7929,  1010,  ...,     0,     0,     0],
        [  101,  2048,  2086,  ...,  3185,  1012,   102],
        [  101,  2092,  1010,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [38]:
bert_encoding["input_ids"]
# Here we can see the padded tokens are returned as 0

tensor([[  101,  2754, 17241,  ...,     0,     0,     0],
        [  101,  1005,  1996,  ...,     0,     0,     0],
        [  101,  7929,  1010,  ...,     0,     0,     0],
        [  101,  2048,  2086,  ...,  3185,  1012,   102],
        [  101,  2092,  1010,  ...,     0,     0,     0]])

In [40]:
bert_encoding["attention_mask"]

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]])

In [43]:
albert_tokenizer = transformers.AutoTokenizer.from_pretrained("albert-base-v2", use_fast = True)
albert_encoding = albert_tokenizer(
    train_reviews[:3], padding=True, truncation=True, max_length=500,
    add_special_tokens=False, return_tensors="pt")
albert_token_ids = albert_encoding["input_ids"]
albert_token_ids

# By deafault albert-base-v2 uses rust to process the tokenization and manually saying use_fast = True ensures it fully owrks on it
# It also uses underscore(_) to seperate words known as Sentencepiece tokenizer instead of wordpiece

tensor([[  876,  5004,    18,   478,    57,    21,   394,  4173,     9,    59,
           478,   340,    70,   699,   101,    21,   171,  3336,    23,  1659,
          1037,    27,    14,   876,    13,     5,  4289,    28,    13,     7,
          4893,   449,     7,     6,     9, 12508,  1612,  5909,    22,    18,
          1400,  8968,    14,   171,  2481,    15,    56,    25,  1118,  1956,
           179,    14,  2151,  1434,    61,    90,   683,  2404,     9,   174,
            15,    32,    22,    18,  2210,    20,   361,    35,    26,    98,
            32,    25,     9,    14,  5427,   128,   832, 22427,    17,  4479,
         24604,    25,  1450,  7472,     9,    14, 12289,    16,    66,  1429,
            50, 12891,     9, 22427,    25, 10356,    28,   550,    15,    17,
         24604,  3049,    53,    16,    33,   310, 11285,    20,   510,   601,
             9,     1,  5145,    13,   118,     1,  5145,    13,   118,     1,
            49, 14586,    30,    31,    22,   195,  

In [41]:
hf_tokenizer = transformers.PreTrainedTokenizerFast(
    tokenizer_object=bpe_tokenizer)
hf_encodings = hf_tokenizer(train_reviews[:3], padding=True, truncation=True,
                            max_length=500, return_tensors="pt")
hf_encodings["input_ids"]

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266